In [10]:
import re

# Regular expression to parse SSH authentication log lines
AUTH_LINE_RE = re.compile(
    r"(?P<result>Accepted|Failed) password for (?P<user>\S+) from "
    r"(?P<ip>[\d.]+) port (?P<port>\d+)"
)


def parse_auth_log(lines):
    """
    Parse raw auth.log lines into structured dictionaries.
    """

    entries = []

    for line in lines:
        match = AUTH_LINE_RE.search(line)

        if match:
            entries.append({
                "result": match.group("result"),
                "user": match.group("user"),
                "ip": match.group("ip"),
                "port": int(match.group("port")),
                "raw": line,
            })

    return entries


def flag_suspicious_logins(entries, trusted_ips):
    """
    Flag successful logins from untrusted IP addresses.
    Root logins are marked HIGH severity.
    """

    flagged = []

    for e in entries:
        if e["result"] == "Accepted" and e["ip"] not in trusted_ips:
            severity = "HIGH" if e["user"] == "root" else "MEDIUM"

            flagged.append({
                **e,
                "severity": severity
            })

    return flagged

In [11]:
log_lines = [
    "Aug 4 10:00:01 server sshd[1234]: Failed password for admin from 192.168.1.100 port 52100 ssh2",
    "Aug 4 10:02:15 server sshd[1235]: Accepted password for root from 203.0.113.5 port 52101 ssh2",
    "Aug 4 10:05:20 server sshd[1236]: Accepted password for user1 from 192.168.1.10 port 52102 ssh2"
]

trusted_ips = {"192.168.1.10"}

entries = parse_auth_log(log_lines)
print(entries)

flagged = flag_suspicious_logins(entries, trusted_ips)
print(flagged)

[{'result': 'Failed', 'user': 'admin', 'ip': '192.168.1.100', 'port': 52100, 'raw': 'Aug 4 10:00:01 server sshd[1234]: Failed password for admin from 192.168.1.100 port 52100 ssh2'}, {'result': 'Accepted', 'user': 'root', 'ip': '203.0.113.5', 'port': 52101, 'raw': 'Aug 4 10:02:15 server sshd[1235]: Accepted password for root from 203.0.113.5 port 52101 ssh2'}, {'result': 'Accepted', 'user': 'user1', 'ip': '192.168.1.10', 'port': 52102, 'raw': 'Aug 4 10:05:20 server sshd[1236]: Accepted password for user1 from 192.168.1.10 port 52102 ssh2'}]
[{'result': 'Accepted', 'user': 'root', 'ip': '203.0.113.5', 'port': 52101, 'raw': 'Aug 4 10:02:15 server sshd[1235]: Accepted password for root from 203.0.113.5 port 52101 ssh2', 'severity': 'HIGH'}]
